# Porosity prediction with GLU-Residual Network (GRN)

Porosity is predicted from post-stack seismic amplitude and the HPSO-inverted acoustic impedance.

Steps:
1. load well trace (seismic amplitude, inverted impedance, porosity log)
2. random split 75 / 10 / 15 (train / validation / blind test)
3. compute seismic attributes of both input traces
4. remove highly correlated attributes (|r| > 0.95) + standard scaling + PCA (99.5 % variance)
5. Bayesian (Optuna TPE) search of hidden_dim, n_blocks, dropout
6. train final model (7 random seeds, predictions averaged)
7. metrics, plots, save
8. (optional) apply on a seismic section

By default it runs on the small **synthetic** example in `example_data/` with `QUICK_TEST = True`.
For the real run, set `DATA_FILES` and `QUICK_TEST = False`.

In [ ]:
import os, time, copy, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import pearsonr
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from torch.optim.swa_utils import AveragedModel, SWALR

import optuna
from optuna.samplers import TPESampler
optuna.logging.set_verbosity(optuna.logging.WARNING)

import joblib
import seismic_attributes as sa

import warnings
warnings.filterwarnings("ignore")

In [ ]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device :", DEVICE)
print("torch", torch.__version__, "| optuna", optuna.__version__)

### settings

In [ ]:
# input file(s) : csv or xlsx, one well trace per file (traces of 201 samples, 1 ms)
DATA_FILES = ["example_data/synthetic_well_trace.csv"]

AMP_COL = "Seismic_Amplitude"
IMP_COL = "Inverted_Impedance_HPSO"
TARGET_COL = "Porosity"           # v/v

N_SAMPLES = 201                   # samples per trace (1700-2100 ms window at 2 ms -> resampled to 201)
OUT_DIR = "outputs/porosity"
RUN_NAME = "porosity_grn"

QUICK_TEST = True                 # True = few trials/epochs only to check the code is working

In [ ]:
VAL_FRAC = 0.10
TEST_FRAC = 0.15
SEED = 42

CORR_THRESHOLD = 0.95
PCA_VARIANCE = 0.995

N_TRIALS = 40          # optuna trials
EPOCHS = 3000
PATIENCE_HPO = 80      # early stopping patience inside each optuna trial
PATIENCE = 150         # early stopping patience for the final training
N_SEEDS = 7            # final model = mean of 7 networks with different seeds

LR = 1e-3
WEIGHT_DECAY = 1e-4
BATCH_SIZE = 32
GRAD_CLIP = 1.0
MIXUP_ALPHA = 0.2
SWA_START = 0.75       # fraction of EPOCHS after which SWA starts

if QUICK_TEST:
    N_TRIALS, EPOCHS, N_SEEDS = 3, 150, 2
    PATIENCE_HPO, PATIENCE = 20, 30
    print("QUICK_TEST is on -> results are only for checking the code !!")

os.makedirs(OUT_DIR, exist_ok=True)

### 1. load data

In [ ]:
feats, targets = [], []
for f in DATA_FILES:
    if f.endswith(".csv"):
        df = pd.read_csv(f)
    else:
        df = pd.read_excel(f)
    df = df.iloc[:N_SAMPLES]
    feats.append(df[[AMP_COL, IMP_COL]].to_numpy(dtype=np.float32))
    targets.append(df[TARGET_COL].to_numpy(dtype=np.float32))
    print(f, df.shape)

X_raw = np.vstack(feats)
y = np.concatenate(targets).astype(np.float64)
n = len(y)
print("samples:", n)

In [ ]:
# quick look at the inputs
fig, ax = plt.subplots(1, 3, figsize=(8, 5), sharey=True)
ax[0].plot(X_raw[:, 0], np.arange(n), 'k', lw=0.8); ax[0].set_title("amplitude")
ax[1].plot(X_raw[:, 1], np.arange(n), 'b', lw=0.8); ax[1].set_title("inv. impedance")
ax[2].plot(y, np.arange(n), 'r', lw=0.8); ax[2].set_title(TARGET_COL)
ax[0].invert_yaxis(); ax[0].set_ylabel("sample")
plt.tight_layout(); plt.show()

### 2. train / val / test split

In [ ]:
def split_random(n_total, val_frac, test_frac, seed=42):
    rng = np.random.default_rng(seed=seed)
    idx = np.arange(n_total)
    rng.shuffle(idx)
    n_test = max(1, int(round(n_total * test_frac)))
    n_val = max(1, int(round(n_total * val_frac)))
    test = np.sort(idx[:n_test])
    val = np.sort(idx[n_test:n_test + n_val])
    train = np.sort(idx[n_test + n_val:])
    return train, val, test

train_idx, val_idx, test_idx = split_random(n, VAL_FRAC, TEST_FRAC, SEED)
print("train", len(train_idx), " val", len(val_idx), " test", len(test_idx))

train_mask = np.zeros(n, bool); train_mask[train_idx] = True
val_mask = np.zeros(n, bool); val_mask[val_idx] = True
test_mask = np.zeros(n, bool); test_mask[test_idx] = True

### 3. seismic attributes
attributes are computed for the amplitude trace (Ch1) and the impedance trace (Ch2)

In [ ]:
t0 = time.time()
X_att, att_names = sa.attribute_matrix(X_raw, N_SAMPLES)
print(X_att.shape, "  %.1f s" % (time.time() - t0))

In [ ]:
# remove nan / constant columns
X_att = np.nan_to_num(X_att, nan=0.0, posinf=0.0, neginf=0.0)
const_keep = np.std(X_att, axis=0) > 1e-10
X_att = X_att[:, const_keep]
att_names = [a for a, k in zip(att_names, const_keep) if k]
print("dropped constant:", (~const_keep).sum(), " -> features:", X_att.shape[1])

### 4. feature reduction (only fitted on the training samples)

In [ ]:
def corr_filter(X, threshold=0.95):
    # for each highly correlated pair drop the one with smaller variance
    c = np.nan_to_num(np.corrcoef(X, rowvar=False))
    up = np.triu(np.abs(c), k=1)
    var = np.var(X, axis=0)
    drop = set()
    for i in range(up.shape[0]):
        for j in range(i + 1, up.shape[1]):
            if up[i, j] > threshold:
                drop.add(i if var[i] < var[j] else j)
    return sorted(set(range(X.shape[1])) - drop)

keep_idx = corr_filter(X_att[train_idx], CORR_THRESHOLD)
print("after correlation filter:", len(keep_idx))

In [ ]:
scaler = StandardScaler()
X_tr_sc = scaler.fit_transform(X_att[train_idx][:, keep_idx]).astype(np.float32)
X_all_sc = scaler.transform(X_att[:, keep_idx]).astype(np.float32)

n_train = len(train_idx)
pca = None
if len(keep_idx) > 0.5 * n_train:
    # still too many features for ~150 samples -> PCA
    max_comp = min(X_tr_sc.shape[1], n_train - 1)
    cum = np.cumsum(PCA(n_components=max_comp).fit(X_tr_sc).explained_variance_ratio_)
    n_pc = min(max(int(np.searchsorted(cum, PCA_VARIANCE) + 1), 2), max_comp)
    pca = PCA(n_components=n_pc).fit(X_tr_sc)
    X_all_s = pca.transform(X_all_sc).astype(np.float32)
    print("PCA components:", n_pc, " variance kept: %.4f" % cum[n_pc - 1])
else:
    X_all_s = X_all_sc

X_train, y_train = X_all_s[train_idx], y[train_idx]
X_val, y_val = X_all_s[val_idx], y[val_idx]
X_test, y_test = X_all_s[test_idx], y[test_idx]
n_in = X_train.shape[1]
print("network input dim:", n_in)

### 5. GLU-Residual network

GLU-ResBlock (eq. 1-4):  h = Dropout(ReLU(W1 x + b1)),  [v, g] = W2 h + b2,  out = LayerNorm(v * sigmoid(g) + proj(x))

VSN (eq. 5-6):  w = softmax(GLU-ResBlock_vsn(x)),  x~ = x * w

stack of n blocks, last block has hidden_dim/2 outputs, then linear head (eq. 7)

In [ ]:
class GRNBlock(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, dropout=0.1):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, output_dim * 2)   # value + gate
        if input_dim != output_dim:
            self.proj = nn.Linear(input_dim, output_dim)
        else:
            self.proj = nn.Identity()
        self.ln = nn.LayerNorm(output_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        res = self.proj(x)
        h = self.dropout(torch.relu(self.fc1(x)))
        v, g = self.fc2(h).chunk(2, dim=-1)
        return self.ln(v * torch.sigmoid(g) + res)


class VariableSelectionNetwork(nn.Module):
    def __init__(self, n_features, hidden_dim):
        super().__init__()
        self.grn = GRNBlock(n_features, hidden_dim, n_features)
        self.softmax = nn.Softmax(dim=-1)

    def forward(self, x):
        w = self.softmax(self.grn(x))
        return x * w


class GatedResidualNet(nn.Module):
    def __init__(self, n_input, hidden_dim=256, n_blocks=4, dropout=0.1):
        super().__init__()
        self.vsn = VariableSelectionNetwork(n_input, hidden_dim)
        layers = []
        prev = n_input
        for i in range(n_blocks):
            out = hidden_dim if i < n_blocks - 1 else hidden_dim // 2
            layers.append(GRNBlock(prev, hidden_dim, out, dropout))
            prev = out
        self.blocks = nn.Sequential(*layers)
        self.head = nn.Linear(prev, 1)

    def forward(self, x):
        return self.head(self.blocks(self.vsn(x))).squeeze(-1)

In [ ]:
# small check
_m = GatedResidualNet(n_in, 64, 3, 0.1)
print(_m(torch.randn(4, n_in)).shape, sum(p.numel() for p in _m.parameters()), "params")
del _m

### training functions
loss = Huber + 0.1*(1 - PCC), AdamW, cosine annealing (warm restarts), mixup, SWA at the end, early stopping on val loss

In [ ]:
def nrmse(yt, yp):
    return np.sqrt(mean_squared_error(yt, yp)) / (yt.max() - yt.min() + 1e-8)


def mixup(x, y, alpha=0.2):
    if alpha <= 0:
        return x, y
    lam = max(np.random.beta(alpha, alpha), 1 - np.random.beta(alpha, alpha))
    idx = torch.randperm(x.size(0), device=x.device)
    return lam * x + (1 - lam) * x[idx], lam * y + (1 - lam) * y[idx]


huber = nn.HuberLoss(delta=1.0)

def loss_fn(pred, target):
    loss = huber(pred, target)
    if len(pred) > 2:
        vx = pred - pred.mean()
        vy = target - target.mean()
        pcc = torch.sum(vx * vy) / (torch.sqrt(torch.sum(vx ** 2) + 1e-8) * torch.sqrt(torch.sum(vy ** 2) + 1e-8))
        loss = loss + 0.1 * (1.0 - pcc)
    return loss

In [ ]:
def predict(model, X):
    model.eval()
    with torch.no_grad():
        return model(torch.tensor(X, dtype=torch.float32).to(DEVICE)).cpu().numpy()


def train_model(model, Xtr, ytr, Xv, yv, epochs=EPOCHS, patience=PATIENCE, lr=LR,
                weight_decay=WEIGHT_DECAY, batch_size=BATCH_SIZE, verbose=True):
    model = model.to(DEVICE)
    bs = min(batch_size, max(8, len(Xtr) // 4))
    tr_ds = TensorDataset(torch.tensor(Xtr, dtype=torch.float32), torch.tensor(ytr, dtype=torch.float32))
    loader = DataLoader(tr_ds, batch_size=bs, shuffle=True, drop_last=False)
    xv_t = torch.tensor(Xv, dtype=torch.float32).to(DEVICE)
    yv_t = torch.tensor(yv, dtype=torch.float32).to(DEVICE)

    opt = optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    sched = optim.lr_scheduler.CosineAnnealingWarmRestarts(opt, T_0=50, T_mult=2, eta_min=lr * 0.01)
    swa_model = AveragedModel(model)
    swa_sched = SWALR(opt, swa_lr=lr * 0.5)
    swa_start = int(epochs * SWA_START)

    best_loss, best_state, wait = np.inf, None, 0
    hist = {"train_loss": [], "val_loss": [], "train_nrmse": [], "val_nrmse": []}
    t0 = time.time()

    for ep in range(epochs):
        model.train()
        tot, nb = 0.0, 0
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            if MIXUP_ALPHA > 0 and ep < swa_start:
                xb, yb = mixup(xb, yb, MIXUP_ALPHA)
            opt.zero_grad(set_to_none=True)
            loss = loss_fn(model(xb), yb)
            loss.backward()
            if GRAD_CLIP > 0:
                nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            opt.step()
            tot += loss.item(); nb += 1

        if ep >= swa_start:
            swa_model.update_parameters(model); swa_sched.step()
        else:
            sched.step()

        model.eval()
        with torch.no_grad():
            pv = model(xv_t)
            vloss = loss_fn(pv, yv_t).item()
        hist["train_loss"].append(tot / max(nb, 1))
        hist["val_loss"].append(vloss)
        hist["train_nrmse"].append(nrmse(ytr, predict(model, Xtr)))
        hist["val_nrmse"].append(nrmse(yv, pv.cpu().numpy()))

        if verbose and (ep + 1) % 100 == 0:
            print(f"  ep {ep+1}  train {tot/max(nb,1):.5f}  val {vloss:.5f}  [{time.time()-t0:.0f}s]")

        # early stopping
        if vloss < best_loss - 1e-6:
            best_loss, wait = vloss, 0
            best_state = copy.deepcopy(model.state_dict())
        else:
            wait += 1
            if wait >= patience:
                if verbose:
                    print(f"  early stop at epoch {ep+1}, best val loss {best_loss:.5f}")
                break

    if best_state is not None:
        model.load_state_dict(best_state)

    # use SWA weights only if they are better on validation
    if ep >= swa_start:
        try:
            torch.optim.swa_utils.update_bn(DataLoader(tr_ds, batch_size=len(tr_ds)), swa_model, device=DEVICE)
            r2_a = r2_score(yv, predict(model, Xv))
            r2_b = r2_score(yv, predict(swa_model, Xv))
            if r2_b > r2_a:
                model.load_state_dict({k.replace("module.", ""): v for k, v in swa_model.state_dict().items()
                                       if k.startswith("module.")})
                if verbose:
                    print("  SWA weights used (val R2 %.4f -> %.4f)" % (r2_a, r2_b))
        except Exception as e:
            print("  swa skipped:", e)

    hist["stopped_epoch"] = ep + 1
    hist["best_epoch"] = int(np.argmin(hist["val_loss"])) + 1
    model.eval()
    return model, hist

### 6. hyperparameter search (Optuna, TPE sampler)
objective = R2 on the validation set

In [ ]:
def objective(trial):
    hd = trial.suggest_categorical("hidden_dim", [128, 192, 256, 384, 512])
    nb = trial.suggest_int("n_blocks", 3, 8)
    dr = trial.suggest_float("dropout", 0.05, 0.3)
    torch.manual_seed(SEED); np.random.seed(SEED)
    m = GatedResidualNet(n_in, hd, nb, dr)
    m, _ = train_model(m, X_train, y_train, X_val, y_val, patience=PATIENCE_HPO, verbose=False)
    return r2_score(y_val, predict(m, X_val))

t0 = time.time()
study = optuna.create_study(direction="maximize", sampler=TPESampler(seed=SEED))
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)
print("HPO time %.1f min" % ((time.time() - t0) / 60))
print("best val R2:", round(study.best_value, 4))
best = study.best_params
print(best)

In [ ]:
trials_df = study.trials_dataframe()
trials_df.to_csv(os.path.join(OUT_DIR, RUN_NAME + "_optuna_trials.csv"), index=False)
trials_df.sort_values("value", ascending=False).head(10)

### 7. final training with the best hyperparameters (7 seeds)

In [ ]:
models, hists = [], []
for k in range(N_SEEDS):
    s = SEED + 1000 * k
    torch.manual_seed(s); np.random.seed(s)
    print(f"--- member {k+1}/{N_SEEDS}  seed {s}")
    m = GatedResidualNet(n_in, best["hidden_dim"], best["n_blocks"], best["dropout"])
    m, h = train_model(m, X_train, y_train, X_val, y_val)
    models.append(m); hists.append(h)

def predict_ens(X):
    return np.mean([predict(m, X) for m in models], axis=0)

### 8. evaluation

In [ ]:
def metrics(yt, yp):
    return {
        "R2": r2_score(yt, yp),
        "PCC": pearsonr(yt, yp)[0],
        "RMSE": np.sqrt(mean_squared_error(yt, yp)),
        "MAE": mean_absolute_error(yt, yp),
        "MAPE_%": np.mean(np.abs((yt - yp) / (np.abs(yt) + 1e-8))) * 100,
        "Bias": np.mean(yp - yt),
        "NRMSE": nrmse(yt, yp),
        "Explained_variance": 1 - np.var(yt - yp) / (np.var(yt) + 1e-8),
    }

y_pred = predict_ens(X_all_s)
res = pd.DataFrame({
    "Train": metrics(y[train_mask], y_pred[train_mask]),
    "Validation": metrics(y[val_mask], y_pred[val_mask]),
    "Test": metrics(y[test_mask], y_pred[test_mask]),
    "Full trace": metrics(y, y_pred),
})
res.to_csv(os.path.join(OUT_DIR, RUN_NAME + "_metrics.csv"))
res.round(4)

In [ ]:
split = np.where(train_mask, "train", np.where(val_mask, "validation", "test"))
pd.DataFrame({"sample": np.arange(n), "split": split, "measured": y, "predicted": y_pred}).to_csv(
    os.path.join(OUT_DIR, RUN_NAME + "_predictions.csv"), index=False)

In [ ]:
# log + crossplot (like Fig. 10)
fig, ax = plt.subplots(1, 2, figsize=(9, 6), gridspec_kw={"width_ratios": [1, 1.4]})
ax[0].plot(y, np.arange(n), 'k', lw=1.2, label="measured")
ax[0].plot(y_pred, np.arange(n), 'r--', lw=1.0, label="predicted")
ax[0].invert_yaxis()
ax[0].set_xlabel("Porosity (v/v)"); ax[0].set_ylabel("sample")
ax[0].legend(loc="lower right")

for msk, c, lab in [(train_mask, "tab:blue", "train"), (val_mask, "tab:orange", "validation"),
                    (test_mask, "tab:green", "blind test")]:
    ax[1].scatter(y[msk], y_pred[msk], s=14, c=c, alpha=0.7, label=lab)
lo, hi = min(y.min(), y_pred.min()), max(y.max(), y_pred.max())
ax[1].plot([lo, hi], [lo, hi], 'k--', lw=0.8)
ax[1].set_xlabel("measured"); ax[1].set_ylabel("predicted")
ax[1].set_title("R2 = %.3f, CC = %.3f" % (res.loc["R2", "Full trace"], res.loc["PCC", "Full trace"]))
ax[1].legend(); ax[1].set_aspect("equal")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, RUN_NAME + "_log_crossplot.png"), dpi=300)
plt.show()

In [ ]:
# NRMSE curve of the first ensemble member (like Fig. S1)
h = hists[0]
plt.figure(figsize=(7, 4))
plt.plot(h["train_nrmse"], label="training")
plt.plot(h["val_nrmse"], label="validation")
plt.axvline(h["best_epoch"] - 1, color="k", ls=":", lw=0.8, label="best epoch %d" % h["best_epoch"])
plt.xlabel("epoch"); plt.ylabel("NRMSE"); plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, RUN_NAME + "_nrmse_curve.png"), dpi=300)
plt.show()
pd.DataFrame({k: v for k, v in h.items() if isinstance(v, list)}).to_csv(
    os.path.join(OUT_DIR, RUN_NAME + "_nrmse_history_member1.csv"), index_label="epoch")

### 9. save model

In [ ]:
torch.save({"state_dicts": [m.state_dict() for m in models],
            "kwargs": dict(n_input=n_in, **best)}, os.path.join(OUT_DIR, RUN_NAME + "_weights.pt"))
joblib.dump({"const_keep": const_keep, "keep_idx": keep_idx, "scaler": scaler, "pca": pca,
             "att_names": att_names, "best_params": best, "n_samples": N_SAMPLES},
            os.path.join(OUT_DIR, RUN_NAME + "_preprocessing.joblib"))
with open(os.path.join(OUT_DIR, RUN_NAME + "_best_params.json"), "w") as f:
    json.dump({"best_params": best, "best_val_r2": study.best_value,
               "best_epochs": [hh["best_epoch"] for hh in hists]}, f, indent=2)
print("saved in", OUT_DIR)

### 10. (optional) prediction on a seismic section
two .npy files of shape (n_traces, 201): seismic amplitude and HPSO impedance of the same traces.
set to None to skip.

In [ ]:
SECTION_AMP = "example_data/synthetic_section_amplitude.npy"
SECTION_IMP = "example_data/synthetic_section_impedance.npy"

In [ ]:
if SECTION_AMP is not None and SECTION_IMP is not None:
    amp_sec = np.load(SECTION_AMP)
    imp_sec = np.load(SECTION_IMP)
    ntr = amp_sec.shape[0]
    Xs_raw = np.stack([amp_sec.reshape(-1), imp_sec.reshape(-1)], axis=1).astype(np.float32)
    Xs, _ = sa.attribute_matrix(Xs_raw, N_SAMPLES)
    Xs = np.nan_to_num(Xs, nan=0.0, posinf=0.0, neginf=0.0)[:, const_keep][:, keep_idx]
    Xs = scaler.transform(Xs).astype(np.float32)
    if pca is not None:
        Xs = pca.transform(Xs).astype(np.float32)
    phi_section = predict_ens(Xs).reshape(ntr, N_SAMPLES)
    np.save(os.path.join(OUT_DIR, RUN_NAME + "_section.npy"), phi_section)

    plt.figure(figsize=(8, 5))
    plt.imshow(phi_section.T, aspect="auto", cmap="jet")
    plt.colorbar(label="Porosity (v/v)")
    plt.xlabel("trace"); plt.ylabel("sample")
    plt.savefig(os.path.join(OUT_DIR, RUN_NAME + "_section.png"), dpi=300)
    plt.show()